# T5 attention-head pruning — Kaggle

**Setup (once):**
1. Upload `t5-head-pruning.zip` as a Dataset: *Create → New Dataset* (Kaggle unzips it), or in this notebook *Add Input → Upload*.
2. Notebook settings (right panel): **Accelerator → GPU T4 x2**, **Internet → On** (needs a phone-verified account).
3. Run cells 1–3 interactively to check everything works. Then click **Save Version → Save & Run All (Commit)** — it runs in the background (you can close the browser) and the `results/` folder appears in the notebook's **Output** tab.

In [ ]:
# 1. copy the project to the writable working dir
import glob, os, shutil
src = os.path.dirname(glob.glob('/kaggle/input/**/run_sweep.py', recursive=True)[0])
shutil.copytree(src, '/kaggle/working/t5-head-pruning', dirs_exist_ok=True)
%cd /kaggle/working/t5-head-pruning
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2. correctness tests (~10 s)
!python tests/test_pruning.py

In [ ]:
# 3. optional 5-minute smoke test, then clean up
!python run_sweep.py --task translation --quick --out smoke && rm -rf smoke

In [ ]:
%%bash
# 4. both sweeps in parallel, one per T4 (logs: log_*.txt)
CUDA_VISIBLE_DEVICES=0 python run_sweep.py --task translation > log_translation.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 python run_sweep.py --task summarization --n_eval 300 > log_summarization.txt 2>&1 &
wait
tail -n 5 log_translation.txt log_summarization.txt

In [ ]:
# 5. efficiency benchmark — runs alone on one GPU so timings aren't disturbed
!CUDA_VISIBLE_DEVICES=0 python run_benchmark.py --task translation --method gradient --all_levels
!CUDA_VISIBLE_DEVICES=0 python run_benchmark.py --task summarization --method gradient --target 0.4

In [ ]:
# 6. figures + tables, and a zip in the Output tab
!python make_plots.py
!cd /kaggle/working && zip -qr results.zip t5-head-pruning/results t5-head-pruning/log_*.txt
from IPython.display import Image, display
for p in sorted(glob.glob('results/*.png')): display(Image(p))